
We're given a DNA sequence, and I want to guess the most likely way it was produced if I assume that different parts (exon, splice site, intron) behave differently. It's like reconstructing a path based on the signs left behind.

 Setting Up the Model
Each state (like Exon or Intron) gives out letters with different likelihoods. There are also certain chances of switching from one state to another. I’m coding those first.

In [7]:

import math

labels = ['E', 'S', 'I']

# What each state tends to emit
emits = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    'S': {'A': 0.05, 'C': 0.0,  'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4,  'C': 0.1,  'G': 0.1,  'T': 0.4}
}

# How states connect to each other
moves = {
    'START': {'E': 1.0},
    'E': {'E': 0.9, 'S': 0.1},
    'S': {'I': 1.0},
    'I': {'I': 0.9, 'STOP': 0.1}
}


##  Kickoff: First Position
Trying every possible starting label and scoring it with the first letter.

In [9]:

def log_safe(x):
    return math.log(x) if x > 0 else float('-inf')


In [11]:

def most_likely_states(dna_seq):
    grid = [{}]
    choices = {}

    for label in labels:
        if label in moves['START']:
            score = log_safe(moves['START'][label]) + log_safe(emits[label].get(dna_seq[0], 0))
            grid[0][label] = score
            choices[label] = [label]


##  Main Loop: Step Through the Sequence
For each position in the DNA, I try all label options and remember the best one I could have reached.

In [15]:
import math

def log_safe(x):
    return math.log(x) if x > 0 else float('-inf')

def most_likely_states(dna_seq):
    labels = ['E', 'S', 'I']

    emits = {
        'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
        'S': {'A': 0.05, 'C': 0.0,  'G': 0.95, 'T': 0.0},
        'I': {'A': 0.4,  'C': 0.1,  'G': 0.1,  'T': 0.4}
    }

    moves = {
        'START': {'E': 1.0},
        'E': {'E': 0.9, 'S': 0.1},
        'S': {'I': 1.0},
        'I': {'I': 0.9, 'STOP': 0.1}
    }

    # Start scoring
    grid = [{}]
    choices = {}

    for label in labels:
        if label in moves['START']:
            score = log_safe(moves['START'][label]) + log_safe(emits[label].get(dna_seq[0], 0))
            grid[0][label] = score
            choices[label] = [label]

    # Step through the sequence
    for pos in range(1, len(dna_seq)):
        grid.append({})
        temp_path = {}

        for curr in labels:
            top_score = float('-inf')
            best_prev = None

            for prev in labels:
                if prev in grid[pos-1] and curr in moves.get(prev, {}):
                    transition = log_safe(moves[prev][curr])
                    emission = log_safe(emits[curr].get(dna_seq[pos], 0))
                    current_score = grid[pos-1][prev] + transition + emission

                    if current_score > top_score:
                        top_score = current_score
                        best_prev = prev

            if best_prev:
                grid[pos][curr] = top_score
                temp_path[curr] = choices[best_prev] + [curr]

        choices = temp_path

    # Finish
    final_score = float('-inf')
    final_choice = None

    for lbl in labels:
        if lbl in grid[-1] and 'STOP' in moves.get(lbl, {}):
            score = grid[-1][lbl] + log_safe(moves[lbl]['STOP'])
            if score > final_score:
                final_score = score
                final_choice = lbl

    return ''.join(choices[final_choice]), round(final_score, 3)

    


Now I just check which final state gives the best full score and backtrack the path.


Using the same DNA as the paper to confirm if the most likely path and score match.

In [19]:

seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
result_path, result_score = most_likely_states(seq)
print("Predicted Path:", result_path)
print("Log Score:", result_score)


Predicted Path: EEEEEEEEEEEEEEEEEESIIIIIII
Log Score: -41.22
